Retomamos o corpus e a pergunta da prática anterior para reordenar com ColBERT os três primeiros candidatos da lista fundida.

## 1. Preparando os chunks

Usamos exatamente os mesmos textos e identificadores da primeira prática. O recorte é pequeno para tornar visível quais trechos entram e saem da etapa de reranking.

In [1]:
corpus = [
    (42, "ECONNRESET na entrega",
     "ECONNRESET indica que a conexão foi encerrada antes de uma resposta HTTP completa. "
     "O emissor não recebeu confirmação do webhook."),
    (43, "Resultado desconhecido",
     "Quando a resposta se perde, o emissor não sabe se o consumidor recebeu ou processou o evento. "
     "A nova tentativa deve considerar que o primeiro envio pode ter produzido efeitos."),
    (44, "Backoff entre tentativas",
     "Backoff exponencial aumenta o intervalo entre retentativas de webhooks "
     "e reduz a pressão sobre um serviço temporariamente instável."),
    (77, "Idempotência e duplicidade",
     "O consumidor registra o event_id antes de executar efeitos. "
     "Se o mesmo evento chegar novamente, ele reconhece a entrega anterior "
     "e evita cobrança ou notificação duplicada."),
    (118, "Falha de socket em HTTP",
     "Uma falha de socket pode interromper uma chamada HTTP em andamento "
     "sem revelar ao cliente o resultado da operação. Registre a tentativa "
     "e investigue o estado antes de repetir a chamada."),
    (205, "Rastreando entregas",
     "Logs de webhooks registram event_id, tentativa, resposta e motivo da próxima ação. "
     "O histórico ajuda a investigar entregas sem confirmação."),
    (301, "Rate limit e capacidade",
     "Uma resposta 429 indica que o consumidor recebeu mais eventos do que consegue processar. "
     "O provedor deve reduzir o ritmo e respeitar a política de espera."),
    (302, "Autenticação da API",
     "A autenticação de uma API valida credenciais e permissões antes de aceitar chamadas. "
     "Tokens expirados precisam ser renovados antes de repetir uma requisição."),
]
chunks = [{"id": id, "title": title, "text": text} for id, title, text in corpus]
print(f"chunks preparados: {len(chunks)}")

chunks preparados: 8


## 2. Indexando os três sinais

BM25 e o modelo denso produzem as representações das buscas iniciais. Separadamente, Jina-ColBERT-v2 produz vários vetores por chunk; guardamos esse terceiro campo no mesmo ponto do Qdrant. Esses vetores são preparados na indexação, antes da consulta, e não reutilizam o vetor único da busca densa. O Qdrant roda em memória, sem serviço externo.

Antes de executar: o Jina-ColBERT-v2 ocupa cerca de 2,2 GB no cache do FastEmbed e tem licença CC BY-NC 4.0. O download acontece no primeiro uso e é reutilizado enquanto o cache for preservado. Aqui ele fica em ~/.cache/fastembed; para escolher outro diretório persistente, defina FASTEMBED_CACHE_DIR. Se o kernel não tiver as dependências, instale qdrant-client==1.15.1, fastembed==0.8.1 e sentence-transformers==5.1.0.

In [2]:
import os
from pathlib import Path

from fastembed import LateInteractionTextEmbedding, SparseTextEmbedding
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

cache_dir = Path(os.environ.get("FASTEMBED_CACHE_DIR", Path.home() / ".cache" / "fastembed"))
cache_dir.mkdir(parents=True, exist_ok=True)
texts = [chunk["text"] for chunk in chunks]

dense_model = SentenceTransformer("intfloat/multilingual-e5-small")
sparse_model = SparseTextEmbedding(
    model_name="Qdrant/bm25", language="portuguese", cache_dir=str(cache_dir)
)
colbert_model = LateInteractionTextEmbedding(
    model_name="jinaai/jina-colbert-v2", cache_dir=str(cache_dir), threads=2
)

dense_vectors = dense_model.encode([f"passage: {text}" for text in texts])
sparse_vectors = list(sparse_model.passage_embed(texts))
colbert_vectors = list(colbert_model.passage_embed(texts, batch_size=2))

client = QdrantClient(":memory:")
collection = "webhooks-hybrid-rrf-colbert"
client.create_collection(
    collection_name=collection,
    vectors_config={
        "dense": models.VectorParams(size=dense_vectors.shape[1], distance=models.Distance.COSINE),
        "colbert": models.VectorParams(
            size=colbert_vectors[0].shape[1],
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM
            ),
            hnsw_config=models.HnswConfigDiff(m=0),
        ),
    },
    sparse_vectors_config={"sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)},
)
client.upsert(
    collection_name=collection,
    points=[
        models.PointStruct(
            id=chunk["id"],
            vector={
                "dense": dense.tolist(),
                "sparse": models.SparseVector(
                    indices=sparse.indices.tolist(), values=sparse.values.tolist()
                ),
                "colbert": colbert.tolist(),
            },
            payload={"title": chunk["title"], "text": chunk["text"]},
        )
        for chunk, dense, sparse, colbert in zip(
            chunks, dense_vectors, sparse_vectors, colbert_vectors
        )
    ],
)

print(f"pontos indexados: {client.count(collection_name=collection, exact=True).count}")
print("representações por ponto: dense + sparse BM25 + multivetor ColBERT")
print(f"dimensão de cada vetor ColBERT: {colbert_vectors[0].shape[1]}")

pontos indexados: 8
representações por ponto: dense + sparse BM25 + multivetor ColBERT
dimensão de cada vetor ColBERT: 128


## 3. Recuperando e combinando candidatos

Repetimos o recorte da primeira prática: quatro resultados por busca e RRF com `k = 60`, usando posições a partir de `#1`. Os scores originais não são somados. O desempate por identificador torna a ordem reproduzível quando dois candidatos têm o mesmo score RRF.

In [3]:
query = "Recebi ECONNRESET ao enviar um webhook. Posso tentar novamente sem duplicar o evento?"
query_dense = dense_model.encode([f"query: {query}"])[0].tolist()
query_sparse_embedding = list(sparse_model.query_embed([query]))[0]
query_sparse = models.SparseVector(
    indices=query_sparse_embedding.indices.tolist(),
    values=query_sparse_embedding.values.tolist(),
)
top_k = 4
dense_hits = client.query_points(collection, query=query_dense, using="dense", limit=top_k).points
sparse_hits = client.query_points(collection, query=query_sparse, using="sparse", limit=top_k).points

sparse_positions = {hit.id: position for position, hit in enumerate(sparse_hits, start=1)}
dense_positions = {hit.id: position for position, hit in enumerate(dense_hits, start=1)}
rrf_k = 60
rrf_scores = {}
for positions in (sparse_positions, dense_positions):
    for chunk_id, position in positions.items():
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (rrf_k + position)
fused_ids = sorted(rrf_scores, key=lambda chunk_id: (-rrf_scores[chunk_id], chunk_id))
candidate_ids = fused_ids[:3]

print("BUSCA LEXICAL:", [hit.id for hit in sparse_hits])
print("BUSCA VETORIAL:", [hit.id for hit in dense_hits])
print("LISTA FUNDIDA (RRF):", fused_ids)
print("CANDIDATOS PARA RERANKING:", candidate_ids)
assert len(candidate_ids) == 3

BUSCA LEXICAL: [43, 42, 77, 301]
BUSCA VETORIAL: [42, 205, 43, 77]
LISTA FUNDIDA (RRF): [42, 43, 77, 205, 301]
CANDIDATOS PARA RERANKING: [42, 43, 77]


## 4. Reordenando o top-3 com ColBERT

Agora geramos os vetores da pergunta. O Qdrant calcula MaxSim apenas para os IDs selecionados pelo RRF, usando os multivetores dos chunks já armazenados. O modelo não precisa codificar de novo cada chunk nesta etapa. O score ColBERT é uma nova escala: serve para ordenar os candidatos deste recorte, não para ser somado ao score RRF.

In [4]:
query_colbert = next(colbert_model.query_embed(query))
reranked_hits = client.query_points(
    collection_name=collection,
    query=query_colbert.tolist(),
    using="colbert",
    query_filter=models.Filter(
        must=[models.HasIdCondition(has_id=candidate_ids)]
    ),
    limit=len(candidate_ids),
).points
assert {hit.id for hit in reranked_hits} == set(candidate_ids)

print("ANTES — RRF")
for position, chunk_id in enumerate(candidate_ids, start=1):
    print(f"#{position} Chunk {chunk_id:03d} | RRF={rrf_scores[chunk_id]:.5f}")
print("\nDEPOIS — Jina-ColBERT-v2")
for position, hit in enumerate(reranked_hits, start=1):
    print(f"#{position} Chunk {hit.id:03d} | MaxSim={hit.score:.4f}")

ANTES — RRF
#1 Chunk 042 | RRF=0.03252
#2 Chunk 043 | RRF=0.03227
#3 Chunk 077 | RRF=0.03150

DEPOIS — Jina-ColBERT-v2
#1 Chunk 042 | MaxSim=19.9177
#2 Chunk 077 | MaxSim=18.5397
#3 Chunk 043 | MaxSim=17.5516


## 5. Conferindo o cálculo do MaxSim

Para cada vetor da pergunta, comparamos todos os vetores de um chunk e guardamos a maior similaridade. A soma desses máximos deve reproduzir o score devolvido pelo Qdrant. Os 32 vetores da pergunta incluem posições internas do modelo; não correspondem necessariamente a 32 palavras visíveis no texto.

In [5]:
import numpy as np

example_hit = reranked_hits[0]
stored_point = client.retrieve(
    collection_name=collection, ids=[example_hit.id], with_vectors=["colbert"]
)[0]
document_multivector = np.asarray(stored_point.vector["colbert"])
similarities = query_colbert @ document_multivector.T
max_per_query_vector = similarities.max(axis=1)
manual_score = float(max_per_query_vector.sum())

print(f"Chunk {example_hit.id:03d}: {query_colbert.shape[0]} vetores da pergunta × "
      f"{document_multivector.shape[0]} vetores do chunk")
print(f"soma dos máximos: {manual_score:.4f}")
print(f"score do Qdrant: {example_hit.score:.4f}")
assert np.isclose(manual_score, example_hit.score, atol=1e-4)

Chunk 042: 32 vetores da pergunta × 33 vetores do chunk
soma dos máximos: 19.9177
score do Qdrant: 19.9177


## 6. O que esta prática mostra

Neste corpus, o `Chunk 042` permanece em primeiro, enquanto o `Chunk 077` passa à frente do `Chunk 043` depois do MaxSim. É um caso executado de reordenação, sem ajuste artificial dos scores. O modelo só pôde comparar os três candidatos que chegaram a essa etapa; um chunk fora do recorte não poderia ser promovido.

A execução também mostra o custo adicional: precisamos gerar e armazenar multivetores de todos os chunks, embora apenas três sejam reavaliados nesta pergunta. Não medimos latência nem qualidade de recuperação aqui. A mudança de posição não demonstra que o trecho promovido é mais útil; para afirmar isso, precisaremos de julgamentos de relevância e de uma avaliação no próximo artigo.